In [ ]:
import requests
import pandas as pd
import time

# 1. Define all 9 key states
locations = {
    'Punjab': {'lat': 30.90, 'lon': 75.85},
    'Haryana': {'lat': 29.05, 'lon': 76.08},
    'Uttar Pradesh': {'lat': 26.84, 'lon': 80.94},
    'Madhya Pradesh': {'lat': 23.25, 'lon': 77.41},
    'Rajasthan': {'lat': 26.91, 'lon': 75.78},
    'Bihar': {'lat': 25.09, 'lon': 85.31},
    'Uttarakhand': {'lat': 30.06, 'lon': 79.01},
    'Gujarat': {'lat': 22.25, 'lon': 71.19},
    'Himachal Pradesh': {'lat': 31.10, 'lon': 77.17}
}

# 2. Adjusted time chunks for 1970 to 2017
time_chunks = [
    ("1970-01-01", "1999-12-31"),
    ("2000-01-01", "2017-12-31")
]

all_weather_data = []

print("Starting 48-year exact match download (1970-2017) for all 9 states.")
print("If the API blocks us (Error 429), the script will auto-pause and resume.\n")

# 3. Loop through states and time chunks with Retry logic
for state, coords in locations.items():
    print(f"Fetching data for {state}...")

    for start, end in time_chunks:
        success = False

        while not success:
            url = (
                f"https://archive-api.open-meteo.com/v1/archive?"
                f"latitude={coords['lat']}&longitude={coords['lon']}&"
                f"start_date={start}&end_date={end}&"
                f"daily=temperature_2m_max,temperature_2m_min,precipitation_sum&"
                f"timezone=Asia/Kolkata"
            )

            response = requests.get(url)

            if response.status_code == 200:
                print(f"  -> Successfully downloaded {start[:4]} to {end[:4]}")
                data = response.json()
                daily = data['daily']

                # Package it into rows
                for i in range(len(daily['time'])):
                    all_weather_data.append({
                        'Date': daily['time'][i],
                        'State': state,
                        'T_max_C': daily['temperature_2m_max'][i],
                        'T_min_C': daily['temperature_2m_min'][i],
                        'Rainfall_mm': daily['precipitation_sum'][i]
                    })
                success = True
                time.sleep(5)  # Polite pause

            elif response.status_code == 429:
                print("  -> [API LIMIT HIT] Error 429. Pausing for 60 seconds to cool down...")
                time.sleep(60)
            else:
                print(f"  -> [FATAL ERROR] Code {response.status_code}. Skipping this chunk.")
                success = True

# 4. Convert and Save
weather_df = pd.DataFrame(all_weather_data)
weather_df['Date'] = pd.to_datetime(weather_df['Date'])

print("\n--- Download Complete! ---")
print(f"Total Rows Collected: {len(weather_df)}")

# Save with the new specific filename
file_name = 'Indian_9States_Daily_Weather_1970_2017.csv'
weather_df.to_csv(file_name, index=False)
print(f"Dataset successfully saved as '{file_name}'")

Starting 48-year exact match download (1970-2017) for all 9 states.
If the API blocks us (Error 429), the script will auto-pause and resume.

Fetching data for Punjab...
  -> Successfully downloaded 1970 to 1999
  -> Successfully downloaded 2000 to 2017
Fetching data for Haryana...
  -> Successfully downloaded 1970 to 1999
  -> Successfully downloaded 2000 to 2017
Fetching data for Uttar Pradesh...
  -> Successfully downloaded 1970 to 1999
  -> [API LIMIT HIT] Error 429. Pausing for 60 seconds to cool down...
  -> Successfully downloaded 2000 to 2017
Fetching data for Madhya Pradesh...
  -> Successfully downloaded 1970 to 1999
  -> Successfully downloaded 2000 to 2017
Fetching data for Rajasthan...
  -> Successfully downloaded 1970 to 1999
  -> [API LIMIT HIT] Error 429. Pausing for 60 seconds to cool down...


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **Phase 2: System Architecture, Crop Alignment & Feature Pruning**

To train a robust and highly accurate Machine Learning model, we must process raw daily weather data into high-resolution biological triggers mapped to localized district yields. This requires Crop Year Alignment, Data Augmentation, and strict Feature Selection.

**1. The "Rabi" Crop Year Alignment:**
Wheat in India is a winter crop. It is sown in November and harvested in April. Therefore, the weather in November 2014 dictates the harvest of April 2015. We filter our daily data to isolate this specific "Growing Season" and shift the November/December weather to the correct harvest year.

**2. Biological Feature Engineering:**
Using the daily data, we extract specific biological triggers:
* `Heat_Stress_Days`: Days > 35°C during the March/April grain-filling stage.
* `Sowing_Rainfall`: Total rain during the crucial November/December germination window.
* `Terminal_Heat_Tmax`: The average maximum temperature right before harvest.
* `Avg_Season_Tmin`: Included because warm nighttime temperatures increase plant respiration and reduce grain weight.

**3. High-Resolution District Expansion:**
We map these engineered biological climate features to the granular historical yield records of **hundreds of individual districts**. This scales our dataset to **over 8,000 highly detailed records**.

**4. Transitioning from Version 1: What We Excluded (and Why):**
To avoid **Multicollinearity** (where redundant features confuse the model and dilute its predictive power), we are explicitly dropping several features used in our Version 1 baseline:
* **Global Macro-Indexes (`ENSO_Index`):** El Niño is a global indicator. Its actual localized impacts (reduced rain or elevated heat) are already captured natively by our local rainfall and temperature columns.
* **Generic Averages (`Avg_Temp`, `Rainfall_mm`):** Replaced by stage-specific metrics. Annual or generic averages ignore timing—rain in November helps wheat, but rain in April destroys it.
* **Derived Indices (`Drought_Index`, `Rainfall_Deviation`, `Is_Drought_Year`, `Heat_Stress_Index`, `Is_Heatwave`):** These are mathematically redundant. The model will naturally learn what a "drought" or "heatwave" looks like by cross-analyzing `Sowing_Rainfall` and `Heat_Stress_Days` against `Yield_Lag1` without needing manually calculated secondary flags.
* **Rolling Historical Averages (`Rainfall_3yr_Avg`, `Temp_3yr_Avg`):** 3-year averages "smooth out" the data. We want our XGBoost model to react sharply to *this year's* sudden weather shocks, not a smoothed historical trend. District infrastructure trends are already captured by `Yield_Lag1`.

In [ ]:
import pandas as pd
import numpy as np

print("Loading raw datasets for Option 1 (District Expansion)...")
yield_df = pd.read_csv("/content/drive/MyDrive/wheat_yield_india/dataset/district_wheat_yield_1970_2017_india.csv")
weather_df = pd.read_csv("/content/drive/MyDrive/wheat_yield_india/dataset/Indian_9States_Daily_Weather_1970_2017.csv")
weather_df['Date'] = pd.to_datetime(weather_df['Date'])

# 1. Yield Data Processing (KEEPING DISTRICTS)
yield_df.rename(columns={
    'State Name': 'State',
    'Dist Name': 'District',
    'WHEAT YIELD (Kg per ha)': 'Yield'
}, inplace=True)

# Sort to calculate lag properly per distinct district
yield_df = yield_df.sort_values(by=['State', 'District', 'Year']).reset_index(drop=True)
yield_df['Yield_Lag1'] = yield_df.groupby(['State', 'District'])['Yield'].shift(1)

# 2. Daily Weather Math
weather_df['DTR'] = weather_df['T_max_C'] - weather_df['T_min_C']
weather_df['Month'] = weather_df['Date'].dt.month

# 3. Crop Year Logic & Season Filtering
def assign_crop_year(date_val):
    if date_val.month in [11, 12]: return date_val.year + 1
    else: return date_val.year

season_weather = weather_df[weather_df['Month'].isin([11, 12, 1, 2, 3, 4])].copy()
season_weather['Crop_Year'] = season_weather['Date'].apply(assign_crop_year)

# 4. Extract Growth-Stage Specific Features (grouped by STATE)
print("Calculating State-Level Climate Features...")
season_weather['Is_Sowing_Month'] = season_weather['Month'].isin([11, 12])
season_weather['Is_Grain_Filling_Month'] = season_weather['Month'].isin([3, 4])

adv_features = season_weather.groupby(['State', 'Crop_Year']).apply(lambda x: pd.Series({
    'Avg_Season_Tmax': x['T_max_C'].mean(),
    'Avg_Season_DTR': x['DTR'].mean(),
    'Total_Season_Rain': x['Rainfall_mm'].sum(),
    'Sowing_Rainfall': x.loc[x['Is_Sowing_Month'], 'Rainfall_mm'].sum(),
    'Terminal_Heat_Tmax': x.loc[x['Is_Grain_Filling_Month'], 'T_max_C'].mean(),
    'Heat_Stress_Days': (x.loc[x['Is_Grain_Filling_Month'], 'T_max_C'] > 35).sum(),
    'Frost_Days': (x['T_min_C'] < 5).sum()
})).reset_index()

# 5. Final Merge: Map State Weather to ALL Districts in that State
print("Mapping State Weather to individual Districts...")
final_dataset = pd.merge(yield_df, adv_features, left_on=['State', 'Year'], right_on=['State', 'Crop_Year'], how='inner')

# Drop redundant columns, drop NA rows caused by the Yield_Lag1 shift
final_dataset.drop(columns=['Crop_Year'], inplace=True)
final_dataset.dropna(subset=['Yield_Lag1'], inplace=True)

# Save the Ultimate Dataset
file_name = "Ultimate_District_AI_Ready_Wheat_Data.csv"
final_dataset.to_csv(file_name, index=False)

print(f"\n--- Fast Track Expansion Complete! ---")
print(f"Dataset Shape: {final_dataset.shape} (Massive increase!)")
display(final_dataset.head(3))
print(f"Saved successfully as: {file_name}")

Loading raw datasets for Option 1 (District Expansion)...
Calculating State-Level Climate Features...


/tmp/ipykernel_177/112265089.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  adv_features = season_weather.groupby(['State', 'Crop_Year']).apply(lambda x: pd.Series({


Mapping State Weather to individual Districts...

--- Fast Track Expansion Complete! ---
Dataset Shape: (8178, 14) (Massive increase!)


,Dist Code,Year,State Code,State,District,Yield,Yield_Lag1,Avg_Season_Tmax,Avg_Season_DTR,Total_Season_Rain,Sowing_Rainfall,Terminal_Heat_Tmax,Heat_Stress_Days,Frost_Days
1,909,1971,2,Bihar,Bhagalpur,2540.97,1144.80,27.850829,13.306630,167.4,0.7,33.334426,27.0,0.0
2,909,1972,2,Bihar,Bhagalpur,1041.59,2540.97,28.699451,13.566484,61.8,1.8,36.367213,44.0,0.0
3,909,1973,2,Bihar,Bhagalpur,1253.81,1041.59,29.834254,12.974586,109.9,42.7,37.219672,44.0,0.0


Saved successfully as: Ultimate_District_AI_Ready_Wheat_Data.csv


In [ ]:
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Load the raw Indian Dataset
file_path = '/content/drive/MyDrive/wheat_yield_india/dataset/final_dataset_version2.csv'
df_india = pd.read_csv(file_path)

# 3. Engineer Equalizer Features (Calculate baselines using the FULL history for better accuracy)
india_baselines = df_india.groupby('Dist Code').agg(
    Historical_Mean_Rain=('Total_Season_Rain', 'mean'),
    Historical_Mean_Tmax=('Avg_Season_Tmax', 'mean')
).reset_index()

df_india = pd.merge(df_india, india_baselines, on='Dist Code', how='left')

# Calculate Deviations
df_india['Rainfall_Deviation'] = df_india['Total_Season_Rain'] - df_india['Historical_Mean_Rain']
df_india['Temperature_Anomaly'] = df_india['Avg_Season_Tmax'] - df_india['Historical_Mean_Tmax']

# 4. Filter for the modern agricultural era (1995 - 2017)
df_india = df_india[(df_india['Year'] >= 1995) & (df_india['Year'] <= 2017)].copy()

# 5. Format to match the USA dataset exactly
df_india['Country'] = 'India'

common_columns = [
    'Country', 'State', 'Dist Code', 'District', 'Year', 'Yield', 'Yield_Lag1',
    'Avg_Season_Tmax', 'Total_Season_Rain', 'Sowing_Rainfall', 'Terminal_Heat_Tmax',
    'Temperature_Anomaly', 'Rainfall_Deviation'
]

df_india_clean = df_india[common_columns].reset_index(drop=True)

# 6. Save the final cleaned and filtered dataset
save_path = '/content/drive/MyDrive/wheat_yield_india/dataset/final_india_wheat_dataset.csv'
df_india_clean.to_csv(save_path, index=False)

print(f"Success! Cleaned and filtered India dataset saved to: {save_path}")
print(f"Total Rows (1995-2017): {len(df_india_clean)}")
print(f"Total Columns: {len(df_india_clean.columns)}")
display(df_india_clean.head())

Mounted at /content/drive
Success! Cleaned and filtered India dataset saved to: /content/drive/MyDrive/wheat_yield_india/dataset/final_india_wheat_dataset.csv
Total Rows (1995-2017): 4002
Total Columns: 13


,Country,State,Dist Code,District,Year,Yield,Yield_Lag1,Avg_Season_Tmax,Total_Season_Rain,Sowing_Rainfall,Terminal_Heat_Tmax,Temperature_Anomaly,Rainfall_Deviation
0,India,Bihar,909,Bhagalpur,1995,1590.69,1643.92,28.420994,60.5,0.4,35.459016,-0.105720,-11.295745
1,India,Bihar,909,Bhagalpur,1996,1287.91,1590.69,28.080769,195.5,146.7,35.839344,-0.445946,123.704255
2,India,Bihar,909,Bhagalpur,1997,1722.88,1287.91,28.040884,42.2,0.0,34.083607,-0.485831,-29.595745
3,India,Bihar,909,Bhagalpur,1998,1836.39,1722.88,26.679006,199.7,111.8,32.913115,-1.847709,127.904255
4,India,Bihar,909,Bhagalpur,1999,1686.75,1836.39,28.929834,36.0,30.9,36.796721,0.403119,-35.795745


In [1]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define exactly where your files are located
input_path = '/content/drive/MyDrive/wheat_yield_india/dataset/final_india_wheat_dataset.csv'
output_path = '/content/drive/MyDrive/wheat_yield_india/dataset/final_india_wheat_dataset_moderate_v2.csv'

# 3. Load and sort the dataset chronologically per district
df = pd.read_csv(input_path)
df = df.sort_values(["Dist Code", "Year"]).reset_index(drop=True)

# 4. Temporal Engineering: Lag 2 and Rolling Statistics
df["Yield_Lag2"] = df.groupby("Dist Code")["Yield"].shift(2)
district_median_yield = df.groupby("Dist Code")["Yield"].transform("median")
df["Yield_Lag2"] = df["Yield_Lag2"].fillna(district_median_yield)

df["Yield_Rolling_Mean3"] = (
    df.groupby("Dist Code")["Yield"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

df["Yield_Rolling_Std3"] = (
    df.groupby("Dist Code")["Yield"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).std().fillna(0))
)

# 5. Localized Climate Anomalies (Deviation from district average)
climate_cols = [
    "Avg_Season_Tmax",
    "Total_Season_Rain",
    "Sowing_Rainfall",
    "Terminal_Heat_Tmax",
    "Temperature_Anomaly",
    "Rainfall_Deviation",
]

for col in climate_cols:
    district_mean = df.groupby("Dist Code")[col].transform("mean")
    df[f"{col}_Anomaly"] = df[col] - district_mean

# 6. Climate Interaction Features
eps = 1e-6
df["Terminal_vs_AvgTemp"] = df["Terminal_Heat_Tmax"] - df["Avg_Season_Tmax"]
df["SowingRain_to_TotalRain"] = df["Sowing_Rainfall"] / (df["Total_Season_Rain"] + 1.0)
df["Rain_per_Temp"] = df["Total_Season_Rain"] / (df["Avg_Season_Tmax"] + eps)

# 7. Drop Yield_Lag1 to force the model to learn weather patterns
if "Yield_Lag1" in df.columns:
    df.drop(columns=["Yield_Lag1"], inplace=True)

# 8. Save the new moderate v2 dataset
df.to_csv(output_path, index=False)
print(f"Success! India moderate dataset v2 saved to:\n{output_path}")

Mounted at /content/drive
Success! India moderate dataset v2 saved to:
/content/drive/MyDrive/wheat_yield_india/dataset/final_india_wheat_dataset_moderate_v2.csv
